BIBLIOTECAS

In [ ]:
import pandas as pd

LEITURA DOS ARQUIVOS DESCRITORES DE ETNIAS E LÍNGUAS - CLASSIFICAÇÕES PRÉVIAS E ATUAIS

In [ ]:
caminho_arquivo = 'BDD_etnias_20250505.xlsx';
df_etnias_descritores_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'BDD_linguas_20240604.xlsx';
df_linguas_descritores_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'descritores.xlsx';
df_etnias_descritores_novacod = pd.read_excel(caminho_arquivo, sheet_name='etnias');
df_linguas_descritores_novacod = pd.read_excel(caminho_arquivo, sheet_name='linguas');

display(df_etnias_descritores_coleta.info());
display(df_linguas_descritores_coleta.info());
display(df_etnias_descritores_novacod.info());
display(df_linguas_descritores_novacod.info());

LEITURA DOS REGISTROS DE PESSOAS INDÍGENAS

In [ ]:
caminho_arquivo = 'indigenas.csv'
df_ind = pd.read_csv(caminho_arquivo, sep=';')
df_ind.info()

RENOMEIA COLUNAS

In [ ]:
df_ind.rename(columns={'desc_etnia_1_coleta': 'txt_etnia_1_coleta'}, inplace=True)
df_ind.rename(columns={'desc_etnia_2_coleta': 'txt_etnia_2_coleta'}, inplace=True)
df_ind.rename(columns={'desc_etnia_3_coleta': 'txt_etnia_3_coleta'}, inplace=True)
df_ind.rename(columns={'desc_lingua_1_coleta': 'txt_lingua_1_coleta'}, inplace=True)
df_ind.rename(columns={'desc_lingua_2_coleta': 'txt_lingua_2_coleta'}, inplace=True)
df_ind.info()

JOIN COM AS TABELAS DESCRITORAS DE COLETA E VIGENTES

In [ ]:
df_ind = pd.merge(df_ind, df_etnias_descritores_coleta, how="left", left_on='cod_etnia_1_coleta', right_on="CODIGO")
df_ind = df_ind.drop(columns='CODIGO')
df_ind.insert(loc=df_ind.columns.get_loc('cod_etnia_1_coleta') + 1, column='desc_cod_etnia_1_coleta', value=df_ind.pop('TEXTO'))

df_ind = pd.merge(df_ind, df_linguas_descritores_coleta, how="left", left_on='cod_lingua_1_coleta', right_on="CODIGO")
df_ind = df_ind.drop(columns='CODIGO')
df_ind.insert(loc=df_ind.columns.get_loc('cod_lingua_1_coleta') + 1, column='desc_cod_lingua_1_coleta', value=df_ind.pop('TEXTO'))

df_ind = pd.merge(df_ind, df_etnias_descritores_novacod, how="left", left_on='cod_etnia_1_novacod', right_on="codigo")
df_ind = df_ind.drop(columns='codigo')
df_ind.insert(loc=df_ind.columns.get_loc('cod_etnia_1_novacod') + 1, column='desc_cod_etnia_1_novacod', value=df_ind.pop('descricao'))

df_ind = pd.merge(df_ind, df_linguas_descritores_novacod, how="left", left_on='cod_lingua_1_novacod', right_on="codigo")
df_ind = df_ind.drop(columns='codigo')
df_ind.insert(loc=df_ind.columns.get_loc('cod_lingua_1_novacod') + 1, column='desc_cod_lingua_1_novacod', value=df_ind.pop('descricao'))

df_ind.info()

COMO OS CAMPOS FORAM PREENCHIDOS DURANTE A COLETA POR MEIO DE SELEÇÃO DA CATEGORIA E REGISTRO DO CÓDIGO OU DE FORMA TEXTUAL LIVRE (EXCLUSIVAS), FAÇO O "NVL" DAS COLUNAS DE DESCRIÇÃO DO CÓDIGO COM AS DE TEXTO LIVRE CRIANDO NOVAS COLUNAS COM TODAS AS ENTRADAS TEXTUAIS DE ETNIA E LÍNGUA

In [ ]:
df_ind.insert(loc=df_ind.columns.get_loc('desc_cod_etnia_1_coleta') + 1, column='txt_etnia_entrada_coleta', value=df_ind['txt_etnia_1_coleta'].fillna(df_ind['desc_cod_etnia_1_coleta']))
df_ind.insert(loc=df_ind.columns.get_loc('desc_cod_lingua_1_coleta') + 1, column='txt_lingua_entrada_coleta', value=df_ind['txt_lingua_1_coleta'].fillna(df_ind['desc_cod_lingua_1_coleta']))
df_ind.info()

EXCLUSÃO DOS REGISTRO SEM INFORMAÇÃO DE LÍNGUA NA COLETA

In [ ]:
df_ind = df_ind[~df_ind.txt_lingua_entrada_coleta.isna()]
df_ind.shape

TRATAMENTO DOS REGISTROS SEM INFORMAÇÃO

In [ ]:
df_ind['desc_cod_etnia_1_novacod'] = df_ind['desc_cod_etnia_1_novacod'].fillna('Não Informado')
df_ind['desc_cod_lingua_1_novacod'] = df_ind['desc_cod_lingua_1_novacod'].fillna('Não Informado')
df_ind.loc[
    df_ind['desc_cod_etnia_1_novacod'].isin(['Não sabe','Mal definida','Não determinada']),
    'desc_cod_etnia_1_novacod'
] = 'Não Informado'
df_ind.loc[
    df_ind['desc_cod_lingua_1_novacod'].isin(['Não sabe','Mal definida','Não determinada']),
    'desc_cod_lingua_1_novacod'
] = 'Não Informado'
df_ind['sobrenome'] = df_ind['sobrenome'].fillna('Não Informado')
df_ind['CD_TI'] = df_ind['CD_TI'].fillna('0')
df_ind['CD_TI'] = df_ind['CD_TI'].astype(int)

SELEÇÃO DOS CAMPOS DE INTERESSE PARA UTILIZAÇÃO EM UM MODELO DE CODIFICAÇÃO PARA APENAS A LÍNGUA PRINCIPAL, ESCOPO DESTA ATIVIDADE

In [ ]:
df_ind = df_ind[['txt_lingua_entrada_coleta','desc_cod_etnia_1_novacod','desc_cod_lingua_1_novacod','cod_setor','tipo_setor','sobrenome','CD_TI','VAL_LATITUDE','VAL_LONGITUDE']]
df_ind.info()

CONVERSÃO DO FORMATO DE LATITUDE E LONGITUDE DE DMS PARA DECIMAL

In [ ]:
# Extrai componentes com regex vetorizado
def extrair_componentes(col):
    return col.str.extract(r'(\d+)\s+(\d+)\s+([\d\.]+)\s+([NSEWO])')

# Aplica ao DataFrame
coords = extrair_componentes(df_ind['VAL_LATITUDE'])
coords = coords.astype({0:float, 1:float, 2:float})

# Calcula decimal
df_ind['VAL_LATITUDE'] = coords[0] + coords[1]/60 + coords[2]/3600

# Aplica sinal
df_ind.loc[coords[3].isin(['S', 'O', 'W']), 'VAL_LATITUDE'] *= -1

# Aplica ao DataFrame
coords = extrair_componentes(df_ind['VAL_LONGITUDE'])
coords = coords.astype({0:float, 1:float, 2:float})

# Calcula decimal
df_ind['VAL_LONGITUDE'] = coords[0] + coords[1]/60 + coords[2]/3600

# Aplica sinal
df_ind.loc[coords[3].isin(['S', 'O', 'W']), 'VAL_LONGITUDE'] *= -1

CONVERSÃO DE TODAS AS STRINGS PARA MAIÚSCULO

In [ ]:
colunas_str = df_ind.select_dtypes(include=['object', 'string']).columns
df_ind[colunas_str] = df_ind[colunas_str].apply(lambda col: col.str.upper())

VERIFICAÇÃO E REMOÇÃO DOS REGISTROS DUPLICADOS

In [ ]:
df_ind.duplicated().sum()
df_ind = df_ind.drop_duplicates()
df_ind.info()

ARMAZENAMENTO DOS DADOS TRATADOS EM UM ARQUIVO CSV

In [ ]:
df_ind.to_csv('df_ind_lingua.csv', index=False, sep=';', encoding='utf-8-sig')